# UP-Fall Camera2 → Skeleton Preprocessing for Kaggle

Notebook này tiền xử lý dữ liệu **UP-Fall Camera2** thành skeleton dataset để huấn luyện **LSTM / Transformer**.

Pipeline:

**Camera2 ZIP → MediaPipe Pose → 17 keypoints × (x, y, visibility) → xử lý missing → interpolation → body centering → torso-scale normalization → sliding window 30 frame → Tag-based labeling → subject-independent split → NPZ**

Thiết kế:
- Chỉ dùng **Camera2**.
- Đọc ảnh trực tiếp từ ZIP, không giải nén toàn bộ.
- MediaPipe Pose trích 33 landmarks rồi chọn 17 điểm.
- `Tag 1–5 = Fall`, `Tag 6–11 = No Fall`, `Tag 20 = Ignore/Unknown`.
- Không rotation normalization.
- Split theo subject, không random sequence.
- Sequence dài 30 frame, stride 15.
- Loại sequence mất pose quá nhiều.

In [26]:
!pip install -q "mediapipe==0.10.35" opencv-python-headless pandas tqdm

In [27]:
# 2. IMPORTS
from pathlib import Path
import io
import json
import re
import zipfile
import warnings

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import mediapipe as mp

warnings.filterwarnings("ignore")
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("Has tasks:", hasattr(mp, "tasks"))

MediaPipe: 0.10.35
NumPy: 2.1.3
OpenCV: 4.14.0
Has tasks: True


In [28]:
# 3. CONFIG

# Sửa 2 đường dẫn này theo tên dataset trên Kaggle
from pathlib import Path

DATASET_BASE = Path(
    "/kaggle/input/datasets/anhoa1/upfall-camera2"
)

LABELS_BASE = Path(
    "/kaggle/input/datasets/anhoa1/upfall-labels"
)

OUTPUT_DIR = Path(
    "/kaggle/working/upfall_preprocessed"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Camera:", DATASET_BASE.exists())
print("Labels:", LABELS_BASE.exists())
print("Output:", OUTPUT_DIR)
for p in DATASET_BASE.iterdir():
    print(" -", p.name)

print("\nLabels root:")
for p in LABELS_BASE.iterdir():
    print(" -", p.name)

CAMERA_ID = 2

MIN_DETECTION_CONFIDENCE = 0.5
MIN_TRACKING_CONFIDENCE = 0.5
VISIBILITY_THRESHOLD = 0.5
MIN_LANDMARK_VISIBILITY = 0.5

SEQUENCE_LENGTH = 30
STRIDE = 15

MAX_MISSING_FRAME_RATIO = 0.30
MIN_LABELED_FRAME_RATIO = 0.70
FALL_RATIO_THRESHOLD = 0.50

FALL_TAGS = {1, 2, 3, 4, 5}
NOFALL_TAGS = {6, 7, 8, 9, 10, 11}
IGNORE_TAGS = {20}

TRAIN_SUBJECTS = set(range(1, 11))   # S1-S10
VAL_SUBJECTS   = {11, 12}            # S11-S12
TEST_SUBJECTS  = set(range(13, 18))  # S13-S17

# None = toàn bộ dataset
MAX_TRIALS = None

SAVE_FRAME_LEVEL = False

print("Output:", OUTPUT_DIR)

Camera: True
Labels: True
Output: /kaggle/working/upfall_preprocessed
 - HAR_UP_images2
 - HAR_UP_images
 - HAR_UP_images3

Labels root:
 - Subject9
 - Subject15
 - Subject6
 - Subject14
 - Subject5
 - Subject7
 - Subject1
 - Subject8
 - Subject3
 - Subject11
 - Subject4
 - Subject17
 - Subject13
 - Subject10
 - Subject12
 - Subject16
 - Subject2
Output: /kaggle/working/upfall_preprocessed


In [29]:
from pathlib import Path
import urllib.request

POSE_MODEL_PATH = Path(
    "/kaggle/working/pose_landmarker_full.task"
)

POSE_MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/pose_landmarker/"
    "pose_landmarker_full/float16/1/"
    "pose_landmarker_full.task"
)

if not POSE_MODEL_PATH.exists():

    print("Downloading Pose Landmarker model...")

    urllib.request.urlretrieve(
        POSE_MODEL_URL,
        POSE_MODEL_PATH
    )

print(
    "Pose model:",
    POSE_MODEL_PATH,
    POSE_MODEL_PATH.exists()
)

Pose model: /kaggle/working/pose_landmarker_full.task True


In [30]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = (
    mp.tasks.vision.PoseLandmarkerOptions
)

VisionRunningMode = (
    mp.tasks.vision.RunningMode
)


options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(
            POSE_MODEL_PATH
        )
    ),

    running_mode=(
        VisionRunningMode.IMAGE
    ),

    num_poses=1,

    min_pose_detection_confidence=0.5,

    min_pose_presence_confidence=0.5,
)


pose_landmarker = (
    PoseLandmarker.create_from_options(
        options
    )
)

print("Pose Landmarker READY")

Pose Landmarker READY


W0000 00:00:1792383092.299227     295 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792383092.346983     295 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## 4. 17 keypoints

Notebook chọn 17 điểm từ MediaPipe Pose:

Nose; Left/Right Shoulder; Left/Right Elbow; Left/Right Wrist; Left/Right Hip; Left/Right Knee; Left/Right Ankle; Left/Right Heel; Left/Right Foot Index.

Mỗi frame có `17 × 3 = 51` feature với `(x, y, visibility)`.

In [31]:
SELECTED_LANDMARKS = [
    0,      # nose

    11,     # left shoulder
    12,     # right shoulder

    13,     # left elbow
    14,     # right elbow

    15,     # left wrist
    16,     # right wrist

    23,     # left hip
    24,     # right hip

    25,     # left knee
    26,     # right knee

    27,     # left ankle
    28,     # right ankle

    29,     # left heel
    30,     # right heel

    31,     # left foot index
    32,     # right foot index
]

LANDMARK_NAMES = [
    "nose",

    "left_shoulder",
    "right_shoulder",

    "left_elbow",
    "right_elbow",

    "left_wrist",
    "right_wrist",

    "left_hip",
    "right_hip",

    "left_knee",
    "right_knee",

    "left_ankle",
    "right_ankle",

    "left_heel",
    "right_heel",

    "left_foot_index",
    "right_foot_index",
]

IDX_LEFT_SHOULDER = 1
IDX_RIGHT_SHOULDER = 2

IDX_LEFT_HIP = 7
IDX_RIGHT_HIP = 8

NUM_KEYPOINTS = 17
NUM_FEATURES = 51

print("Keypoints:", NUM_KEYPOINTS)
print("Features/frame:", NUM_FEATURES)

Keypoints: 17
Features/frame: 51


In [32]:
# ============================================================
# 6. UTILITIES
# ============================================================

CAMERA2_DIR_RE = re.compile(
    r"Subject(\d+)Activity(\d+)Trial(\d+)Camera2$",
    re.IGNORECASE
)

LABEL_RE = re.compile(
    r"Subject(\d+)Activity(\d+)Trial(\d+)\.csv$",
    re.IGNORECASE
)

IMAGE_EXTS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
)


def natural_key(text):
    return [
        int(x) if x.isdigit()
        else x.lower()
        for x in re.split(r"(\d+)", text)
    ]


def find_camera2_trials(root):
    """
    Tìm các thư mục Camera2 dạng:

    Subject1/
      Activity1/
        Trial1/
          Subject1Activity1Trial1Camera2/
             *.png
    """

    rows = []

    for folder in root.rglob("*"):

        if not folder.is_dir():
            continue

        m = CAMERA2_DIR_RE.fullmatch(
            folder.name
        )

        if not m:
            continue

        info = {
            "subject": int(m.group(1)),
            "activity": int(m.group(2)),
            "trial": int(m.group(3)),
            "camera": 2,
        }

        image_paths = [
            p
            for p in folder.iterdir()
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTS
        ]

        image_paths.sort(
            key=lambda p:
            natural_key(p.name)
        )

        rows.append({
            "info": info,
            "folder": folder,
            "images": image_paths,
        })

    rows.sort(
        key=lambda x: (
            x["info"]["subject"],
            x["info"]["activity"],
            x["info"]["trial"]
        )
    )

    return rows


def label_from_tag(tag):

    try:
        tag = int(tag)

    except Exception:
        return -1

    if tag in FALL_TAGS:
        return 1

    if tag in NOFALL_TAGS:
        return 0

    return -1


def split_from_subject(subject):

    if subject in TRAIN_SUBJECTS:
        return "train"

    if subject in VAL_SUBJECTS:
        return "val"

    if subject in TEST_SUBJECTS:
        return "test"

    return "unused"

In [33]:
# 7. BUILD LABEL INDEX

# ============================================================
# 7. BUILD LABEL INDEX
# ============================================================

LABEL_RE = re.compile(
    r"Subject(\d+)Activity(\d+)Trial(\d+)\.csv$",
    re.IGNORECASE
)


def build_label_index(labels_base):

    index = {}

    for csv_path in labels_base.rglob("*.csv"):

        m = LABEL_RE.search(
            csv_path.name
        )

        if not m:
            continue

        key = (
            int(m.group(1)),
            int(m.group(2)),
            int(m.group(3)),
        )

        index[key] = csv_path

    return index


LABEL_INDEX = build_label_index(
    LABELS_BASE
)

print(
    "Label CSV files found:",
    len(LABEL_INDEX)
)

if LABEL_INDEX:

    first_key = next(
        iter(LABEL_INDEX)
    )

    print(
        "Example:",
        first_key,
        "->",
        LABEL_INDEX[first_key]
    )


Label CSV files found: 558
Example: (2, 4, 3) -> /kaggle/input/datasets/anhoa1/upfall-labels/Subject2/Activity4/Subject2Activity4Trial3.csv


In [34]:
def read_label_csv(
    subject,
    activity,
    trial
):

    key = (
        subject,
        activity,
        trial
    )

    csv_path = LABEL_INDEX.get(
        key
    )

    if csv_path is None:
        return None

    df = pd.read_csv(
        csv_path
    )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    colmap = {
        c.lower(): c
        for c in df.columns
    }

    tag_col = colmap.get(
        "tag"
    )

    ts_col = (
        colmap.get("timestamp")
        or colmap.get("timestamps")
    )

    if tag_col is None:
        raise ValueError(
            f"Không có cột Tag: {csv_path}"
        )

    out = pd.DataFrame()

    out["tag"] = pd.to_numeric(
        df[tag_col],
        errors="coerce"
    )

    out["frame_label"] = (
        out["tag"]
        .apply(label_from_tag)
    )

    if ts_col is not None:

        out["timestamp"] = (
            pd.to_datetime(
                df[ts_col],
                errors="coerce"
            )
        )

    else:

        out["timestamp"] = pd.NaT

    return out

## 9. Đồng bộ frame với label

Notebook ưu tiên **timestamp alignment** nếu tên frame chứa timestamp parse được. Nếu không, notebook dùng **proportional index alignment** giữa thứ tự frame và thứ tự label row.

Metadata `alignment_method` sẽ được lưu để bạn kiểm tra lại sau.

In [35]:
# 10. TIMESTAMP PARSING + LABEL ALIGNMENT

ISO_TS_RE = re.compile(
    r"(\d{4})[-_](\d{2})[-_](\d{2})[T _-]"
    r"(\d{2})[-_:](\d{2})[-_:](\d{2})(?:[._-](\d{1,6}))?"
)

def parse_timestamp_from_filename(name):
    base = Path(name).stem
    m = ISO_TS_RE.search(base)
    if not m:
        return pd.NaT

    year, month, day, hh, mm, ss, frac = m.groups()
    frac = (frac or "0").ljust(6, "0")[:6]

    s = f"{year}-{month}-{day} {hh}:{mm}:{ss}.{frac}"
    return pd.to_datetime(s, errors="coerce")

def align_labels_to_frames(frame_names, label_df):
    n_frames = len(frame_names)

    if label_df is None or len(label_df) == 0:
        return np.full(n_frames, -1, dtype=np.int8), "missing_label"

    frame_ts = pd.Series([
        parse_timestamp_from_filename(n)
        for n in frame_names
    ])

    valid_frame_ts = frame_ts.notna().sum()
    valid_label_ts = label_df["timestamp"].notna().sum()

    # 1) Timestamp alignment
    if valid_frame_ts >= max(3, int(0.8 * n_frames)) and valid_label_ts > 0:
        labels_valid = (
            label_df.dropna(subset=["timestamp"])
                    .sort_values("timestamp")
        )

        left = pd.DataFrame({
            "frame_idx": np.arange(n_frames),
            "timestamp": frame_ts
        }).dropna(subset=["timestamp"]).sort_values("timestamp")

        right = labels_valid[["timestamp", "frame_label"]]

        merged = pd.merge_asof(
            left,
            right,
            on="timestamp",
            direction="nearest"
        )

        arr = np.full(n_frames, -1, dtype=np.int8)
        arr[merged["frame_idx"].to_numpy(dtype=int)] = (
            merged["frame_label"].fillna(-1).astype(np.int8).to_numpy()
        )

        # Nếu có một ít filename không parse được timestamp,
        # fill chúng theo nearest valid frame index.
        s = pd.Series(arr.astype(float))
        s[s < 0] = np.nan
        s = s.interpolate(method="nearest", limit_direction="both")
        arr = s.fillna(-1).astype(np.int8).to_numpy()

        return arr, "timestamp_nearest"

    # 2) Proportional index alignment
    label_arr = (
        label_df["frame_label"]
        .fillna(-1)
        .astype(np.int8)
        .to_numpy()
    )

    if len(label_arr) == 1:
        return np.repeat(label_arr[0], n_frames), "index_repeat"

    pos = np.linspace(0, len(label_arr) - 1, num=n_frames)
    idx = np.rint(pos).astype(int)
    arr = label_arr[idx]

    return arr.astype(np.int8), "index_proportional"

In [36]:
def extract_skeleton_from_frame(
    frame_bgr,
    landmarker
):

    rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = landmarker.detect(
        mp_image
    )

    if (
        result.pose_landmarks is None
        or len(result.pose_landmarks) == 0
    ):

        return (
            np.full(
                (NUM_KEYPOINTS, 3),
                np.nan,
                dtype=np.float32
            ),
            False
        )

    landmarks = result.pose_landmarks[0]

    skel = np.zeros(
        (NUM_KEYPOINTS, 3),
        dtype=np.float32
    )

    for i, mp_idx in enumerate(
        SELECTED_LANDMARKS
    ):

        lm = landmarks[mp_idx]

        x = float(lm.x)
        y = float(lm.y)

        # Tasks API có visibility
        v = float(
            lm.visibility
            if lm.visibility is not None
            else 0.0
        )

        if v < MIN_LANDMARK_VISIBILITY:

            skel[i, 0] = np.nan
            skel[i, 1] = np.nan

        else:

            skel[i, 0] = x
            skel[i, 1] = y

        skel[i, 2] = v

    return skel, True

In [37]:
# 12. INTERPOLATE MISSING LANDMARKS

def interpolate_skeleton_sequence(frames):
    '''
    frames: (T,17,3)

    x/y: linear interpolation theo thời gian.
    visibility: NaN -> 0.
    '''
    arr = frames.copy().astype(np.float32)

    T, K, C = arr.shape

    for k in range(K):
        for c in [0, 1]:
            s = pd.Series(arr[:, k, c])
            s = s.interpolate(
                method="linear",
                limit_direction="both"
            )
            arr[:, k, c] = s.to_numpy(dtype=np.float32)

    arr[:, :, 2] = np.nan_to_num(
        arr[:, :, 2],
        nan=0.0
    )

    return arr

In [38]:
# 13. BODY-CENTERED + TORSO-SCALE NORMALIZATION

def normalize_skeleton(frames, eps=1e-6):
    '''
    Center:
        midpoint của 2 hips.

    Scale:
        shoulder-center đến hip-center.
        fallback: shoulder width, sau đó body bbox.

    Không rotation normalization.
    '''
    arr = frames.copy().astype(np.float32)

    xy = arr[:, :, :2]
    vis = arr[:, :, 2:3]

    for t in range(arr.shape[0]):
        pts = xy[t]

        lhip = pts[IDX_LEFT_HIP]
        rhip = pts[IDX_RIGHT_HIP]
        lsho = pts[IDX_LEFT_SHOULDER]
        rsho = pts[IDX_RIGHT_SHOULDER]

        hip_center = (lhip + rhip) / 2.0
        shoulder_center = (lsho + rsho) / 2.0

        scale = np.linalg.norm(shoulder_center - hip_center)

        if (not np.isfinite(scale)) or scale < eps:
            scale = np.linalg.norm(lsho - rsho)

        if (not np.isfinite(scale)) or scale < eps:
            valid = pts[np.isfinite(pts).all(axis=1)]
            if len(valid) >= 2:
                mins = valid.min(axis=0)
                maxs = valid.max(axis=0)
                scale = np.linalg.norm(maxs - mins)

        if (not np.isfinite(scale)) or scale < eps:
            scale = 1.0

        xy[t] = (pts - hip_center) / scale

    out = np.concatenate([xy, vis], axis=-1)

    out = np.nan_to_num(
        out,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return out.astype(np.float32)

In [39]:
# 14. WINDOW LABELING

def label_window(frame_labels):
    frame_labels = np.asarray(frame_labels)

    valid = frame_labels >= 0
    labeled_ratio = valid.mean()

    if labeled_ratio < MIN_LABELED_FRAME_RATIO:
        return None, labeled_ratio, np.nan

    valid_labels = frame_labels[valid]

    if len(valid_labels) == 0:
        return None, labeled_ratio, np.nan

    fall_ratio = (valid_labels == 1).mean()
    y = 1 if fall_ratio >= FALL_RATIO_THRESHOLD else 0

    return y, labeled_ratio, fall_ratio

In [40]:
# ============================================================
# 15. PROCESS ONE CAMERA2 IMAGE FOLDER
# ============================================================

def process_trial(image_paths, info):

    subject = info["subject"]
    activity = info["activity"]
    trial = info["trial"]

    # ========================================================
    # READ LABEL
    # ========================================================

    label_df = read_label_csv(
        subject,
        activity,
        trial
    )

    if label_df is None:
        return [], {
            **info,
            "status": "missing_label",
            "num_frames": 0,
            "pose_detection_rate": np.nan,
            "alignment_method": "missing_label",
            "num_sequences": 0,
        }

    # ========================================================
    # SORT IMAGE PATHS
    # ========================================================

    image_paths = sorted(
        image_paths,
        key=lambda p: natural_key(p.name)
    )

    frame_names = [
        p.name
        for p in image_paths
    ]

    if len(frame_names) == 0:
        return [], {
            **info,
            "status": "no_images",
            "num_frames": 0,
            "pose_detection_rate": np.nan,
            "alignment_method": "none",
            "num_sequences": 0,
        }

    # ========================================================
    # ALIGN LABELS TO CAMERA FRAMES
    # ========================================================

    frame_labels, alignment_method = (
        align_labels_to_frames(
            frame_names,
            label_df
        )
    )

    # ========================================================
    # EXTRACT MEDIAPIPE SKELETONS
    # ========================================================

    skeletons = []
    pose_flags = []

    for image_path in image_paths:

        try:

            frame = cv2.imread(
                str(image_path)
            )

            # Image decode failed
            if frame is None:

                skeletons.append(
                    np.full(
                        (
                            NUM_KEYPOINTS,
                            3
                        ),
                        np.nan,
                        dtype=np.float32
                    )
                )

                pose_flags.append(
                    False
                )

                continue

            # MediaPipe Pose
            skel, detected = (
                extract_skeleton_from_frame(
                    frame,
                    pose_landmarker
                )
            )

            skeletons.append(
                skel
            )

            pose_flags.append(
                detected
            )

        except Exception as e:

            skeletons.append(
                np.full(
                    (
                        NUM_KEYPOINTS,
                        3
                    ),
                    np.nan,
                    dtype=np.float32
                )
            )

            pose_flags.append(
                False
            )

    # ========================================================
    # CONVERT TO NUMPY
    # ========================================================

    skeletons = np.asarray(
        skeletons,
        dtype=np.float32
    )

    pose_flags = np.asarray(
        pose_flags,
        dtype=bool
    )

    # ========================================================
    # POSE DETECTION RATE
    # ========================================================

    if len(pose_flags) > 0:

        pose_detection_rate = float(
            pose_flags.mean()
        )

    else:

        pose_detection_rate = 0.0

    # ========================================================
    # CREATE SEQUENCES
    # ========================================================

    samples = []

    for start in range(
        0,
        len(skeletons)
        - SEQUENCE_LENGTH
        + 1,
        STRIDE
    ):

        end = (
            start
            + SEQUENCE_LENGTH
        )

        # ----------------------------------------------------
        # Skeleton sequence
        # ----------------------------------------------------

        seq_raw = skeletons[
            start:end
        ].copy()

        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        seq_labels = frame_labels[
            start:end
        ]

        # ----------------------------------------------------
        # Pose flags
        # ----------------------------------------------------

        seq_pose = pose_flags[
            start:end
        ]

        # ====================================================
        # QUALITY FILTER
        # ====================================================

        missing_frame_ratio = (
            1.0
            - float(
                seq_pose.mean()
            )
        )

        if (
            missing_frame_ratio
            > MAX_MISSING_FRAME_RATIO
        ):
            continue

        # ====================================================
        # WINDOW LABEL
        # ====================================================

        (
            y,
            labeled_ratio,
            fall_ratio
        ) = label_window(
            seq_labels
        )

        if y is None:
            continue

        # ====================================================
        # INTERPOLATE MISSING KEYPOINTS
        # ====================================================

        seq = (
            interpolate_skeleton_sequence(
                seq_raw
            )
        )

        # ====================================================
        # NORMALIZE SKELETON
        # ====================================================

        seq = normalize_skeleton(
            seq
        )

        # ====================================================
        # CHECK SHAPE
        #
        # (30, 17, 3)
        # ->
        # (30, 51)
        # ====================================================

        seq = seq.reshape(
            SEQUENCE_LENGTH,
            NUM_FEATURES
        )

        # ====================================================
        # STORE SAMPLE
        # ====================================================

        samples.append({

            "X": seq.astype(
                np.float32
            ),

            "y": int(y),

            "subject": subject,

            "activity": activity,

            "trial": trial,

            "camera": 2,

            "start_frame": int(
                start
            ),

            "end_frame": int(
                end - 1
            ),

            "missing_frame_ratio": float(
                missing_frame_ratio
            ),

            "labeled_frame_ratio": float(
                labeled_ratio
            ),

            "fall_ratio": float(
                fall_ratio
            ),

            "alignment_method":
                alignment_method,
        })

    # ========================================================
    # TRIAL REPORT
    # ========================================================

    report = {

        **info,

        "status": "ok",

        "num_frames": int(
            len(skeletons)
        ),

        "pose_detection_rate": float(
            pose_detection_rate
        ),

        "alignment_method":
            alignment_method,

        "num_sequences": int(
            len(samples)
        ),
    }

    # ========================================================
    # OPTIONAL FRAME-LEVEL SAVE
    # ========================================================

    if SAVE_FRAME_LEVEL:

        out = OUTPUT_DIR / (
            f"S{subject:02d}_"
            f"A{activity:02d}_"
            f"T{trial:02d}_"
            f"Camera2_frames.npz"
        )

        np.savez_compressed(

            out,

            skeletons=skeletons,

            pose_flags=pose_flags,

            frame_labels=frame_labels,
        )

    return samples, report

In [41]:
if not DATASET_BASE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dataset: {DATASET_BASE}"
    )

all_camera_trials = find_camera2_trials(
    DATASET_BASE
)

print(
    "Total Camera2 trials found:",
    len(all_camera_trials)
)

camera_trials = all_camera_trials

if MAX_TRIALS is not None:
    camera_trials = camera_trials[
        :MAX_TRIALS
    ]

print(
    "Trials selected:",
    len(camera_trials)
)

if camera_trials:

    print(
        "First:",
        camera_trials[0]["info"]
    )

    print(
        "Images:",
        len(
            camera_trials[0]["images"]
        )
    )

    print(
        "Folder:",
        camera_trials[0]["folder"]
    )

Total Camera2 trials found: 559
Trials selected: 559
First: {'subject': 1, 'activity': 1, 'trial': 1, 'camera': 2}
Images: 195
Folder: /kaggle/input/datasets/anhoa1/upfall-camera2/HAR_UP_images/Subject1/Activity1/Trial1/Subject1Activity1Trial1Camera2


In [42]:
print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

print(
    "Camera2 trials found:",
    len(camera_trials)
)

print(
    "Label CSVs:",
    len(LABEL_INDEX)
)

camera_subjects = sorted({
    x["info"]["subject"]
    for x in camera_trials
})

label_subjects = sorted({
    x[0]
    for x in LABEL_INDEX.keys()
})

print(
    "Camera subjects:",
    camera_subjects
)

print(
    "Label subjects:",
    label_subjects
)

if camera_trials:

    first = camera_trials[0]

    print("\nFirst Camera2 trial:")
    print(first["info"])

    print(
        "Images:",
        len(first["images"])
    )

    print(
        "First image:",
        first["images"][0]
    )

DATASET CHECK
Camera2 trials found: 559
Label CSVs: 558
Camera subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Label subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

First Camera2 trial:
{'subject': 1, 'activity': 1, 'trial': 1, 'camera': 2}
Images: 195
First image: /kaggle/input/datasets/anhoa1/upfall-camera2/HAR_UP_images/Subject1/Activity1/Trial1/Subject1Activity1Trial1Camera2/2018-07-04T12_04_17.738369.png


In [43]:
# ============================================================
# CAMERA2 IMAGE CHECK + DATASET CHECK
# ============================================================

from collections import Counter

print("=" * 70)
print("CAMERA2 IMAGE CHECK")
print("=" * 70)

# Tìm toàn bộ Camera2 trial folders
all_camera_trials = find_camera2_trials(
    DATASET_BASE
)

print(
    "Total Camera2 trials found:",
    len(all_camera_trials)
)

subject_counts = Counter()

valid_trials = []
empty_trials = []

total_images = 0


for item in tqdm(
    all_camera_trials,
    desc="Checking Camera2 folders"
):

    info = item["info"]
    images = item["images"]
    folder = item["folder"]

    subject_counts[
        info["subject"]
    ] += 1

    num_images = len(images)

    total_images += num_images

    if num_images == 0:

        empty_trials.append(
            str(folder)
        )

    else:

        valid_trials.append(
            str(folder)
        )


print()
print("=" * 70)
print("RESULT")
print("=" * 70)

print(
    "Total Camera2 trials :",
    len(all_camera_trials)
)

print(
    "Valid trials         :",
    len(valid_trials)
)

print(
    "Empty trials         :",
    len(empty_trials)
)

print(
    "Total images         :",
    total_images
)

print()
print("Trials per subject:")

for subject in sorted(
    subject_counts
):

    print(
        f"Subject {subject:02d}: "
        f"{subject_counts[subject]} trials"
    )


if empty_trials:

    print()
    print("EMPTY CAMERA2 FOLDERS:")

    for p in empty_trials:
        print(" -", p)


# ============================================================
# DATASET CHECK
# ============================================================

print()
print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

# camera_trials dùng để preprocess
camera_trials = find_camera2_trials(
    DATASET_BASE
)

print(
    "Camera2 trials:",
    len(camera_trials)
)

print(
    "Label CSVs:",
    len(LABEL_INDEX)
)


camera_subjects = sorted({
    item["info"]["subject"]
    for item in camera_trials
})

label_subjects = sorted({
    key[0]
    for key in LABEL_INDEX.keys()
})


print(
    "Camera subjects:",
    camera_subjects
)

print(
    "Label subjects:",
    label_subjects
)


# ============================================================
# CAMERA ↔ LABEL MATCH CHECK
# ============================================================

camera_keys = {
    (
        item["info"]["subject"],
        item["info"]["activity"],
        item["info"]["trial"],
    )
    for item in camera_trials
}

label_keys = set(
    LABEL_INDEX.keys()
)


camera_without_label = (
    camera_keys
    - label_keys
)

label_without_camera = (
    label_keys
    - camera_keys
)


print()
print("=" * 70)
print("CAMERA ↔ LABEL CHECK")
print("=" * 70)

print(
    "Camera trials       :",
    len(camera_keys)
)

print(
    "Label trials        :",
    len(label_keys)
)

print(
    "Camera without label:",
    len(camera_without_label)
)

print(
    "Label without Camera2:",
    len(label_without_camera)
)


if camera_without_label:

    print()
    print(
        "Camera trials without label:"
    )

    for key in sorted(
        camera_without_label
    ):
        print(" -", key)


if label_without_camera:

    print()
    print(
        "Labels without Camera2:"
    )

    for key in sorted(
        label_without_camera
    ):
        print(" -", key)


# ============================================================
# SAMPLE TRIAL
# ============================================================

if camera_trials:

    first = camera_trials[0]

    print()
    print("=" * 70)
    print("FIRST CAMERA2 TRIAL")
    print("=" * 70)

    print(
        "Info:",
        first["info"]
    )

    print(
        "Folder:",
        first["folder"]
    )

    print(
        "Images:",
        len(
            first["images"]
        )
    )

    if first["images"]:

        print(
            "First image:",
            first["images"][0]
        )

        print(
            "Last image:",
            first["images"][-1]
        )

CAMERA2 IMAGE CHECK
Total Camera2 trials found: 559


Checking Camera2 folders:   0%|          | 0/559 [00:00<?, ?it/s]


RESULT
Total Camera2 trials : 559
Valid trials         : 559
Empty trials         : 0
Total images         : 294681

Trials per subject:
Subject 01: 33 trials
Subject 02: 33 trials
Subject 03: 33 trials
Subject 04: 33 trials
Subject 05: 33 trials
Subject 06: 33 trials
Subject 07: 33 trials
Subject 08: 31 trials
Subject 09: 33 trials
Subject 10: 33 trials
Subject 11: 33 trials
Subject 12: 33 trials
Subject 13: 33 trials
Subject 14: 33 trials
Subject 15: 33 trials
Subject 16: 33 trials
Subject 17: 33 trials

DATASET CHECK
Camera2 trials: 559
Label CSVs: 558
Camera subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Label subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

CAMERA ↔ LABEL CHECK
Camera trials       : 559
Label trials        : 558
Camera without label: 1
Label without Camera2: 0

Camera trials without label:
 - (6, 10, 2)

FIRST CAMERA2 TRIAL
Info: {'subject': 1, 'activity': 1, 'trial': 1, 'camera': 2}
Folder: /kaggle/input/datasets/anho

In [44]:
camera_keys = set()
for item in all_camera_trials:

    info = item["info"]

    camera_keys.add(
        (
            info["subject"],
            info["activity"],
            info["trial"]
        )
    )


label_keys = set(LABEL_INDEX.keys())


camera_without_label = (
    camera_keys - label_keys
)

label_without_camera = (
    label_keys - camera_keys
)


print("=" * 70)
print("CAMERA ↔ LABEL CHECK")
print("=" * 70)

print("Camera trials :", len(camera_keys))
print("Label trials  :", len(label_keys))

print(
    "Camera without label:",
    len(camera_without_label)
)

print(
    "Label without Camera2:",
    len(label_without_camera)
)


if camera_without_label:
    print("\nCamera trials missing label:")

    for item in sorted(camera_without_label):
        print(" -", item)


if label_without_camera:
    print("\nLabels without Camera2:")

    for item in sorted(label_without_camera):
        print(" -", item)

CAMERA ↔ LABEL CHECK
Camera trials : 559
Label trials  : 558
Camera without label: 1
Label without Camera2: 0

Camera trials missing label:
 - (6, 10, 2)


In [45]:
# 17. RUN PREPROCESSING

all_samples = []
trial_reports = []

for item in tqdm(
    camera_trials,
    desc="Processing Camera2 trials"
):

    info = item["info"]
    image_paths = item["images"]
    try:
        samples, report =process_trial(image_paths,info)
        all_samples.extend(samples)
        trial_reports.append(report)

    except Exception as e:
        trial_reports.append({
            **info,
            "status": f"error:{type(e).__name__}",
            "num_frames": 0,
            "pose_detection_rate": np.nan,
            "alignment_method": "error",
            "num_sequences": 0,
        })

        print("\nERROR:", zip_path)
        print(e)

report_df = pd.DataFrame(trial_reports)
report_path = OUTPUT_DIR / "trial_preprocessing_report.csv"
report_df.to_csv(report_path, index=False)

print("\nTrials processed:", len(trial_reports))
print("Sequences created:", len(all_samples))
print("Report:", report_path)

Processing Camera2 trials:   0%|          | 0/559 [00:00<?, ?it/s]

W0000 00:00:1792384563.447205     296 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.



Trials processed: 559
Sequences created: 17239
Report: /kaggle/working/upfall_preprocessed/trial_preprocessing_report.csv


In [60]:
# 18. BUILD DATASET ARRAYS

if len(all_samples) == 0:
    raise RuntimeError(
        "Không tạo được sequence nào. "
        "Kiểm tra path, labels hoặc quality thresholds."
    )

X = np.stack([s["X"] for s in all_samples]).astype(np.float32)
y = np.asarray([s["y"] for s in all_samples], dtype=np.int8)

subject = np.asarray([s["subject"] for s in all_samples], dtype=np.int16)
activity = np.asarray([s["activity"] for s in all_samples], dtype=np.int8)
trial = np.asarray([s["trial"] for s in all_samples], dtype=np.int8)
camera = np.asarray([s["camera"] for s in all_samples], dtype=np.int8)

start_frame = np.asarray([s["start_frame"] for s in all_samples], dtype=np.int32)
end_frame = np.asarray([s["end_frame"] for s in all_samples], dtype=np.int32)

missing_frame_ratio = np.asarray(
    [s["missing_frame_ratio"] for s in all_samples],
    dtype=np.float32
)

labeled_frame_ratio = np.asarray(
    [s["labeled_frame_ratio"] for s in all_samples],
    dtype=np.float32
)

fall_ratio = np.asarray(
    [s["fall_ratio"] for s in all_samples],
    dtype=np.float32
)

alignment_method = np.asarray(
    [s["alignment_method"] for s in all_samples],
    dtype=str
)

print("X:", X.shape)
print("y:", y.shape)
print("Fall:", int((y == 1).sum()))
print("No Fall:", int((y == 0).sum()))

X: (17239, 30, 51)
y: (17239,)
Fall: 470
No Fall: 16769


In [61]:
# 19. SUBJECT-INDEPENDENT SPLIT

split = np.asarray(
    [split_from_subject(int(s)) for s in subject],
    dtype=str
)

train_mask = split == "train"
val_mask = split == "val"
test_mask = split == "test"

print("Train:", train_mask.sum(), "subjects:", sorted(set(subject[train_mask].tolist())))
print("Val  :", val_mask.sum(), "subjects:", sorted(set(subject[val_mask].tolist())))
print("Test :", test_mask.sum(), "subjects:", sorted(set(subject[test_mask].tolist())))

assert set(subject[train_mask]).isdisjoint(set(subject[val_mask]))
assert set(subject[train_mask]).isdisjoint(set(subject[test_mask]))
assert set(subject[val_mask]).isdisjoint(set(subject[test_mask]))

print("No subject leakage.")

Train: 10407 subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Val  : 2046 subjects: [11, 12]
Test : 4786 subjects: [13, 14, 15, 16, 17]
No subject leakage.


In [62]:
# 20. DATASET SUMMARY

summary_rows = []

for split_name, mask in [
    ("train", train_mask),
    ("val", val_mask),
    ("test", test_mask),
]:
    n = int(mask.sum())

    summary_rows.append({
        "split": split_name,
        "samples": n,
        "fall": int(((y == 1) & mask).sum()),
        "no_fall": int(((y == 0) & mask).sum()),
        "subjects": ",".join(
            map(str, sorted(set(subject[mask].tolist())))
        ),
        "mean_missing_ratio": (
            float(missing_frame_ratio[mask].mean())
            if n else np.nan
        ),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False
)

,split,samples,fall,no_fall,subjects,mean_missing_ratio
0,train,10407,314,10093,"1,2,3,4,5,6,7,8,9,10",0.009164
1,val,2046,56,1990,"11,12",0.015152
2,test,4786,100,4686,"13,14,15,16,17",0.019536


In [63]:
# 21. SAVE FULL DATASET

full_path = OUTPUT_DIR / "upfall_camera2_skeleton_17kp_30f_full.npz"

np.savez_compressed(
    full_path,
    X=X,
    y=y,
    subject=subject,
    activity=activity,
    trial=trial,
    camera=camera,
    start_frame=start_frame,
    end_frame=end_frame,
    missing_frame_ratio=missing_frame_ratio,
    labeled_frame_ratio=labeled_frame_ratio,
    fall_ratio=fall_ratio,
    alignment_method=alignment_method,
    split=split,
)

print("Saved:", full_path)
print("Size MB:", full_path.stat().st_size / (1024**2))

Saved: /kaggle/working/upfall_preprocessed/upfall_camera2_skeleton_17kp_30f_full.npz
Size MB: 43.82046031951904


In [64]:
# 22. SAVE TRAIN / VAL / TEST

def save_split(name, mask):
    out = OUTPUT_DIR / f"upfall_{name}.npz"

    np.savez_compressed(
        out,
        X=X[mask],
        y=y[mask],
        subject=subject[mask],
        activity=activity[mask],
        trial=trial[mask],
        start_frame=start_frame[mask],
        end_frame=end_frame[mask],
        missing_frame_ratio=missing_frame_ratio[mask],
        labeled_frame_ratio=labeled_frame_ratio[mask],
        fall_ratio=fall_ratio[mask],
        alignment_method=alignment_method[mask],
    )

    print(name, "->", out, "|", X[mask].shape)

save_split("train", train_mask)
save_split("val", val_mask)
save_split("test", test_mask)

train -> /kaggle/working/upfall_preprocessed/upfall_train.npz | (10407, 30, 51)
val -> /kaggle/working/upfall_preprocessed/upfall_val.npz | (2046, 30, 51)
test -> /kaggle/working/upfall_preprocessed/upfall_test.npz | (4786, 30, 51)


In [65]:
# 23. CLASS WEIGHTS

train_y = y[train_mask]

n0 = int((train_y == 0).sum())
n1 = int((train_y == 1).sum())
n = len(train_y)

class_weight = {
    0: n / (2 * n0) if n0 > 0 else 1.0,
    1: n / (2 * n1) if n1 > 0 else 1.0,
}

print("Train No Fall:", n0)
print("Train Fall   :", n1)
print("Class weight :", class_weight)

with open(
    OUTPUT_DIR / "class_weight.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(class_weight, f, indent=2)

Train No Fall: 10093
Train Fall   : 314
Class weight : {0: 0.5155553353809571, 1: 16.571656050955415}


In [66]:
# 24. FINAL VALIDATION

print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("X dtype:", X.dtype)
print("X shape:", X.shape)
print("Expected sample shape:", (SEQUENCE_LENGTH, NUM_FEATURES))

print("NaN count:", int(np.isnan(X).sum()))
print("Inf count:", int(np.isinf(X).sum()))

print("Unique labels:", np.unique(y, return_counts=True))
print("Subjects:", sorted(set(subject.tolist())))
print(
    "Alignment methods:",
    pd.Series(alignment_method).value_counts().to_dict()
)

assert X.ndim == 3
assert X.shape[1] == SEQUENCE_LENGTH
assert X.shape[2] == NUM_FEATURES
assert np.isnan(X).sum() == 0
assert np.isinf(X).sum() == 0
assert set(np.unique(y)).issubset({0, 1})

print("\nDataset READY for LSTM / Transformer.")

FINAL VALIDATION
X dtype: float32
X shape: (17239, 30, 51)
Expected sample shape: (30, 51)
NaN count: 0
Inf count: 0
Unique labels: (array([0, 1], dtype=int8), array([16769,   470]))
Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Alignment methods: {'timestamp_nearest': 17239}

Dataset READY for LSTM / Transformer.


## Output

Sau khi chạy xong:

```text
/kaggle/working/upfall_preprocessed/
├── upfall_camera2_skeleton_17kp_30f_full.npz
├── upfall_train.npz
├── upfall_val.npz
├── upfall_test.npz
├── trial_preprocessing_report.csv
├── dataset_summary.csv
└── class_weight.json
```

Tensor:

```text
X: (N, 30, 51)
y: (N,)
```

Split:

```text
Train: Subject 1–10
Val:   Subject 11–12
Test:  Subject 13–17
```

Nếu `alignment_method = index_proportional` xuất hiện nhiều, cần kiểm tra lại naming convention của ảnh trong ZIP. Nếu filename có timestamp theo format khác, nên bổ sung parser để đồng bộ frame ↔ label chính xác hơn.

In [67]:
import numpy as np
import pandas as pd

full_path = (
    "/kaggle/working/upfall_preprocessed/"
    "upfall_camera2_skeleton_17kp_30f_full.npz"
)

data = np.load(
    full_path,
    allow_pickle=True
)

df = pd.DataFrame({
    "y": data["y"],
    "subject": data["subject"],
    "activity": data["activity"],
    "trial": data["trial"],
})

print("=== ALL ===")
print(
    df.groupby(
        ["activity", "y"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print("\n=== FALL ONLY ===")
print(
    df[df["y"] == 1]
    .groupby("activity")
    .size()
)

=== ALL ===
y            0    1
activity           
1          365   93
2          200   93
3          323  109
4          241   67
5          301  108
6         3269    0
7         3521    0
8         3489    0
9          511    0
10        1675    0
11        2874    0

=== FALL ONLY ===
activity
1     93
2     93
3    109
4     67
5    108
dtype: int64
